In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/fraud_oracle_with_telematics.csv")
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

Shape: (15420, 43)

Columns:
 ['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year', 'BasePolicy', 'avg_speed_kmph', 'max_speed_kmph', 'hard_brakes_per_trip', 'rapid_acceleration_events', 'trip_duration_minutes', 'distance_km', 'night_driving_ratio', 'urban_driving_ratio', 'harsh_cornering_events', 'idle_time_minutes']


In [2]:
# Step 1: Normalize column names and text values
df.columns = df.columns.str.strip().str.lower()

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [3]:
# Step 2: Handle missing values
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

cat_cols = df.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values after fill: {df.isnull().sum().sum()}")

Missing values after fill: 0


In [4]:
# Step 3: Map range-based columns to numeric values
range_mappings = {
    'days_policy_claim': {'none': 0, '1 to 7': 3, '8 to 15': 10, '15 to 30': 20, 'more than 30': 40},
    'days_policy_accident': {'none': 0, '1 to 7': 3, '8 to 15': 10, '15 to 30': 20, 'more than 30': 40},
    'pastnumberofclaims': {'none': 0, '1': 1, '2 to 4': 3, 'more than 4': 5},
    'numberofsuppliments': {'none': 0, '1 to 2': 1.5, '3 to 5': 4, 'more than 5': 6}
}

for col, mapping in range_mappings.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)
        df[col] = pd.to_numeric(df[col])

In [5]:
# Step 4: Create derived telematics features
if 'max_speed_kmph' in df.columns:
    df['speeding_risk'] = (df['max_speed_kmph'] > 120).astype(int)
    df['speed_volatility'] = (df['max_speed_kmph'] - df.get('avg_speed_kmph', df['max_speed_kmph'])).abs()

if 'hard_brakes_per_trip' in df.columns:
    df['harsh_braking_risk'] = (df['hard_brakes_per_trip'] > 5).astype(int)

if 'rapid_acceleration_events' in df.columns:
    df['harsh_acceleration_risk'] = (df['rapid_acceleration_events'] > 5).astype(int)

if 'harsh_cornering_events' in df.columns:
    df['harsh_cornering_risk'] = (df['harsh_cornering_events'] > 5).astype(int)

behavior_cols = ['hard_brakes_per_trip', 'rapid_acceleration_events', 'harsh_cornering_events']
if all(col in df.columns for col in behavior_cols):
    df['harsh_driving_index'] = (df['hard_brakes_per_trip'] + df['rapid_acceleration_events'] + df['harsh_cornering_events']) / 3

if 'night_driving_ratio' in df.columns:
    df['high_night_driving'] = (df['night_driving_ratio'] > 0.5).astype(int)

if 'urban_driving_ratio' in df.columns:
    df['high_urban_driving'] = (df['urban_driving_ratio'] > 0.7).astype(int)

In [6]:
# Step 5: Feature Engineering - Claim Features
if 'days_policy_claim' in df.columns:
    df['fast_claim'] = (df['days_policy_claim'] < 7).astype(int)

if 'pastnumberofclaims' in df.columns:
    df['high_claim_history'] = (df['pastnumberofclaims'] > 2).astype(int)

if 'fast_claim' in df.columns and 'harsh_driving_index' in df.columns:
    df['claim_driving_risk'] = df['fast_claim'] * df['harsh_driving_index']

In [7]:
# Step 5a: Remove ID leakage columns
id_cols = ['policynumber', 'repnumber']
for col in id_cols:
    if col in df.columns:
        df = df.drop(col, axis=1)

# Step 5b: Convert binary yes/no columns to numeric
binary_yes_no = ['policereportfiled', 'witnesspresent']
for col in binary_yes_no:
    if col in df.columns:
        df[col] = df[col].map({'yes': 1, 'no': 0})

# Step 5c: Label-encode categorical columns
cat_for_model = ['policytype', 'vehiclecategory', 'vehicleprice', 'ageofvehicle', 'ageofpolicyholder', 'numberofcars']
cat_present = [col for col in cat_for_model if col in df.columns]

from sklearn.preprocessing import LabelEncoder
for col in cat_present:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [8]:
# Step 6: Identify numeric columns for scaling
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [9]:
# Step 7: Standard Scaling for numeric features
scaler = StandardScaler()
exclude_from_scaling = ['fraudfound_p', 'year']
scale_cols = [col for col in numeric_cols if col not in exclude_from_scaling]

df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [10]:
# Step 8: Final validation
print("=== PREPROCESSING CHECK ===")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Fraud rate: {(df['fraudfound_p'] == 1).sum() / len(df):.2%}")

=== PREPROCESSING CHECK ===
Shape: (15420, 52)
Missing values: 0
Fraud rate: 5.99%


In [11]:
# Display sample
df.head()

,month,weekofmonth,dayofweek,make,accidentarea,dayofweekclaimed,monthclaimed,weekofmonthclaimed,sex,maritalstatus,...,speed_volatility,harsh_braking_risk,harsh_acceleration_risk,harsh_cornering_risk,harsh_driving_index,high_night_driving,high_urban_driving,fast_claim,high_claim_history,claim_driving_risk
0,dec,1.717545,wednesday,honda,urban,tuesday,jan,-1.345408,female,single,...,0.725376,-0.301255,-0.524215,-0.134501,-0.340056,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
1,jan,0.164199,wednesday,honda,urban,monday,jan,1.037295,male,single,...,-0.317649,-0.301255,1.907613,-0.134501,1.661724,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
2,oct,1.717545,friday,honda,urban,thursday,nov,-0.551174,male,married,...,1.399306,-0.301255,-0.524215,-0.134501,-0.006426,1.010037,1.139325,-0.008053,-0.972492,-0.008053
3,jun,-0.612473,saturday,toyota,rural,friday,jul,-1.345408,male,married,...,-0.821335,-0.301255,1.907613,-0.134501,0.994464,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
4,jan,1.717545,monday,honda,urban,tuesday,feb,-0.551174,female,single,...,0.170197,-0.301255,-0.524215,-0.134501,-1.340946,-0.990062,1.139325,-0.008053,-0.972492,-0.008053


In [12]:
# Save processed data
df.to_csv("../data/processed.csv", index=False)
print("✓ Processed data saved to: data/processed.csv")

✓ Processed data saved to: data/processed.csv


In [13]:
# Verify target variable
print("Target distribution:")
print(df['fraudfound_p'].value_counts())

Target distribution:
fraudfound_p
0    14497
1      923
Name: count, dtype: int64


In [14]:
# Final categorical encoding
remaining_str_cols = df.select_dtypes(include='object').columns.tolist()

if remaining_str_cols:
    from sklearn.preprocessing import LabelEncoder
    for col in remaining_str_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    df.to_csv("../data/processed.csv", index=False)

In [ ]:
# Final summary
print("\n" + "=" * 80)
print("✓ PREPROCESSING COMPLETE - LSTM,GNN and CATBOOST READY")
print("=" * 80)
print(f"Shape: {df.shape} | Fraud rate: {(df['fraudfound_p'] == 1).sum() / len(df):.2%}")
print(f"Features: 38 numeric (scaled) + 12 categorical (encoded)")
print(f"Output: data/processed.csv")
print("=" * 80)


✓ PREPROCESSING COMPLETE - GNN and CATBOOST READY
Shape: (15420, 52) | Fraud rate: 5.99%
Features: 38 numeric (scaled) + 12 categorical (encoded)
Output: data/processed.csv
